In [2]:
# ============================================================
# 📦 Cell 1：環境準備（首次執行 or 環境有變動時才需要跑這格）
# ============================================================
import os
import urllib.request

PROJECT_DIR = r"c:\Users\user\Documents\PowerQuery\Gemma4_ComfyUI"
COMFYUI_DIR = os.path.join(PROJECT_DIR, "ComfyUI")

os.chdir(PROJECT_DIR)
print("📂 工作目錄：", PROJECT_DIR)

# 1. 取得 ComfyUI
if not os.path.exists(COMFYUI_DIR):
    print("📦 正在下載 ComfyUI...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    print("✅ ComfyUI 資料夾已存在。")

# 2. 建立虛擬環境 & 安裝套件
os.chdir(COMFYUI_DIR)
if not os.path.exists(".venv"):
    print("🔧 正在建立虛擬環境...")
    !uv venv
print("📦 正在安裝/更新套件...")
!uv pip install --python .venv -q torch torchvision torchaudio
!uv pip install --python .venv -q -r requirements.txt

# 3. 下載生圖模型
checkpoints_dir = os.path.join(COMFYUI_DIR, "models", "checkpoints")
os.makedirs(checkpoints_dir, exist_ok=True)
model_path = os.path.join(checkpoints_dir, "DreamShaper_8_pruned.safetensors")
model_url = "https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors"

if not os.path.exists(model_path):
    print("⏳ 正在下載生圖模型 (約 2GB)...")
    def _progress(bn, bs, ts):
        if ts > 0:
            print(f"\r   下載進度 {min(bn*bs,ts)*100/ts:.1f}%", end="")
    urllib.request.urlretrieve(model_url, model_path, _progress)
    print("\n✅ 模型下載完成！")
else:
    print("✅ 模型已存在！")

os.chdir(PROJECT_DIR)
print("\n🎉 環境準備完成！請執行下一格來啟動服務。")


📂 工作目錄： c:\Users\user\Documents\PowerQuery\Gemma4_ComfyUI
✅ ComfyUI 資料夾已存在。
📦 正在安裝/更新套件...
✅ 模型已存在！

🎉 環境準備完成！請執行下一格來啟動服務。


In [1]:
# ============================================================
# 🚀 Cell 2：一鍵啟動 ComfyUI + Open WebUI（每次要用就跑這格）
# ============================================================
import subprocess
import time
import webbrowser
import socket
import os

PROJECT_DIR = r"c:\Users\user\Documents\PowerQuery\Gemma4_ComfyUI"
COMFYUI_DIR = os.path.join(PROJECT_DIR, "ComfyUI")
VENV_PYTHON = os.path.join(COMFYUI_DIR, ".venv", "Scripts", "python.exe")
OPEN_WEBUI  = r"C:\Users\user\.local\bin\open-webui.exe"

CREATE_NEW_CONSOLE = 0x00000010

# ── 函式定義 ──
def kill_port(port):
    """清理佔用指定 Port 的程序"""
    try:
        out = subprocess.check_output(
            f'netstat -ano | findstr :{port}', shell=True
        ).decode('utf-8', errors='ignore')
        for line in out.splitlines():
            if 'LISTENING' in line:
                pid_str = line.strip().split()[-1]
                subprocess.call(
                    ['taskkill', '/F', '/PID', pid_str],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
                )
                print(f"   🧹 已清理 Port {port} (PID: {pid_str})")
    except subprocess.CalledProcessError:
        pass  # Port 沒有被佔用，正常

def wait_for_port(port, label, max_wait=90):
    """等待指定 Port 可連線，每 2 秒檢查一次"""
    print(f"   ⏳ 等待 {label} (Port {port}) 啟動中", end="", flush=True)
    for i in range(max_wait // 2):
        try:
            with socket.create_connection(('127.0.0.1', port), timeout=1):
                print(f" ✅ 就緒！(花了 {i*2} 秒)")
                return True
        except OSError:
            print(".", end="", flush=True)
            time.sleep(2)
    print(f" ⚠️ 超時（{max_wait}秒），請手動檢查黑色視窗。")
    return False

# ── Step 1: 清理舊程序 ──
print("🧹 Step 1/4：清理之前殘留的服務...")
kill_port(8188)
kill_port(8080)
time.sleep(1)

# ── Step 2: 啟動 ComfyUI ──
print("🚀 Step 2/4：啟動 ComfyUI 引擎（黑色視窗）...")
comfyui_full_cmd = f'title ComfyUI Server && cd /d "{COMFYUI_DIR}" && "{VENV_PYTHON}" main.py --cpu'
subprocess.Popen(
    f'cmd.exe /k "{comfyui_full_cmd}"',
    creationflags=CREATE_NEW_CONSOLE
)

# ── Step 3: 啟動 Open WebUI ──
print("🚀 Step 3/4：啟動 Open WebUI（黑色視窗）...")
webui_full_cmd = f'title Open WebUI Server && cd /d "{PROJECT_DIR}" && "{OPEN_WEBUI}" serve'
subprocess.Popen(
    f'cmd.exe /k "{webui_full_cmd}"',
    creationflags=CREATE_NEW_CONSOLE
)

# ── Step 4: 等待服務就緒 ──
print("📡 Step 4/4：等待伺服器就緒...")
comfy_ok = wait_for_port(8188, "ComfyUI")
webui_ok = wait_for_port(8080, "Open WebUI")

# ── 完成 ──
print()
print("🟢" * 30)
if comfy_ok and webui_ok:
    print("🎉 兩個服務都已成功啟動！")
else:
    print("⚠️ 部分服務可能還在啟動中，請稍後重整網頁。")
print()
print("🌎 ComfyUI 介面：  http://127.0.0.1:8188")
print("🌎 Open WebUI：    http://127.0.0.1:8080")
print("👉 請在 Open WebUI [設定] → [圖像] 確認 ComfyUI 基礎 URL 為 http://127.0.0.1:8188")
print("🟢" * 30)

webbrowser.open("http://127.0.0.1:8080")
print("\n💡 提示：請勿關閉工具列上的兩個黑色 CMD 視窗！那就是正在運行的服務。")


🧹 Step 1/4：清理之前殘留的服務...
🚀 Step 2/4：啟動 ComfyUI 引擎（黑色視窗）...
🚀 Step 3/4：啟動 Open WebUI（黑色視窗）...
📡 Step 4/4：等待伺服器就緒...
   ⏳ 等待 ComfyUI (Port 8188) 啟動中..... ✅ 就緒！(花了 10 秒)
   ⏳ 等待 Open WebUI (Port 8080) 啟動中.... ✅ 就緒！(花了 8 秒)

🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢
🎉 兩個服務都已成功啟動！

🌎 ComfyUI 介面：  http://127.0.0.1:8188
🌎 Open WebUI：    http://127.0.0.1:8080
👉 請在 Open WebUI [設定] → [圖像] 確認 ComfyUI 基礎 URL 為 http://127.0.0.1:8188
🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢

💡 提示：請勿關閉工具列上的兩個黑色 CMD 視窗！那就是正在運行的服務。


In [ ]:
# ============================================================
# 🛑 Cell 3：停止所有服務（不用時跑這格來關閉）
# ============================================================
import subprocess

def kill_port(port):
    try:
        out = subprocess.check_output(
            f'netstat -ano | findstr :{port}', shell=True
        ).decode('utf-8', errors='ignore')
        for line in out.splitlines():
            if 'LISTENING' in line:
                pid_str = line.strip().split()[-1]
                subprocess.call(
                    ['taskkill', '/F', '/PID', pid_str],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
                )
                print(f"   🧹 已關閉 Port {port} (PID: {pid_str})")
    except subprocess.CalledProcessError:
        print(f"   ✅ Port {port} 沒有程序在執行")

print("🛑 正在關閉所有服務...")
kill_port(8188)
kill_port(8080)
print("✅ 所有服務已停止！")
